In [1]:
import os
import math
import json
import torch
import random
import torchaudio
import torchaudio.transforms as T
import numpy as np
from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from sklearn.metrics import *
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


<h2>Preprocessing Function</h2>

In [2]:
TARGET_SR=16000
MAX_LEN=TARGET_SR*4

def preprocess_audio(path):
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0]>1:
        waveform=waveform.mean(dim=0, keepdim=True)
    if sr!=TARGET_SR:
        waveform=T.Resample(sr,TARGET_SR)(waveform)
    if waveform.shape[1]>MAX_LEN:
        waveform=waveform[:, :MAX_LEN]
    else:
        waveform=torch.nn.functional.pad(
            waveform,(0,MAX_LEN-waveform.shape[1])
        )
    return waveform.squeeze(0)

'def get_embedding(waveform_1d, feature_extractor, encoder, device):\n    #extracts w2v2 embedding from preprocessed 1d waveform and return tensor of shape [768]\n    inputs=feature_extractor(\n        waveform_1d.numpy(),sampling_rate=TARGET_SR,return_tensors="pt",padding=True\n    )\n    with torch.no_grad():\n        outputs=encoder(inputs.input_values.to(device))\n    embedding=outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu()\n    assert embedding.shape==torch.Size([768]),         f"Unexpected embedding shape: {embedding.shape}"\n    return embedding\nprint("function defined")'

In [3]:
from transformers import Wav2Vec2Model
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
class W2VDataset(Dataset):
    def __init__(self, protocol,flac_dir,augment=False):
        self.flac_dir=flac_dir
        self.augment=augment
        self.samples=[]

        with open(protocol) as f:
            for line in f:
                parts=line.strip().split()
                fname=parts[1]
                label=0 if parts[4] == "bonafide" else 1
                self.samples.append((fname,label))

    def _augment(self,wav):
        if random.random()<0.5:
            noise=torch.randn_like(wav)*0.01
            wav=wav+noise
        return wav

    def __getitem__(self,idx):
        fname,label=self.samples[idx]
        path=os.path.join(self.flac_dir,fname+".flac")
        wav=preprocess_audio(path)
        if self.augment:
            wav=self._augment(wav)
        return wav,torch.tensor(label,dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

<h2>File Paths</h2>

In [9]:
train_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac"
dev_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac"
eval_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac"
train_protocol="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocol="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocol="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"

paths = {
    "train_flac": train_flac_dir,
    "dev_flac": dev_flac_dir,
    "eval_flac": eval_flac_dir,
    "train_protocol": train_protocol,
    "dev_protocol": dev_protocol,
    "eval_protocol": eval_protocol
}
all_ok=True
for name, path in paths.items():
    exists = os.path.exists(path)
    print(f"{name}: {'OK' if exists else 'MISSING'} — {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll paths verified.")
    print("Train flac:",len([f for f in os.listdir(train_flac_dir) if f.endswith(".flac")]))
    print("Dev flac:",len([f for f in os.listdir(dev_flac_dir) if f.endswith(".flac")]))
    print("Eval flac:",len([f for f in os.listdir(eval_flac_dir) if f.endswith(".flac")]))
else:
    print("\nFix missing paths before continuing.")

train_flac: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac
dev_flac: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac
eval_flac: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac
train_protocol: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
dev_protocol: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
eval_protocol: OK — /kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt

All paths verified.
Train flac: 25380
Dev flac: 24986
Eval flac: 71933


In [10]:
class W2VModel(nn.Module):
    def __init__(self):
        super().__init__()
        #self.feature_extractor=Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
        self.encoder = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        for name, param in self.encoder.named_parameters():
            if "encoder.layers.10" not in name and \
                "encoder.layers.11" not in name:
                param.requires_grad=False
        self.attention=nn.Sequential(
            nn.Linear(768,128),
            nn.Tanh(),
            nn.Linear(128,1)
        )
        self.classifier=nn.Sequential(
            nn.Linear(768,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,1)
        )
    def forward(self,wav):
        input_values = (wav - wav.mean(dim=-1, keepdim=True)) / \
                   (wav.std(dim=-1, keepdim=True) + 1e-7)
        #input_values=inputs.input_values.to(device)
        outputs=self.encoder(input_values)
        hidden=outputs.last_hidden_state
        attn=torch.softmax(self.attention(hidden),dim=1)
        pooled=torch.sum(hidden*attn,dim=1)
        return self.classifier(pooled)

In [11]:
train_dataset=W2VDataset(train_protocol, train_flac_dir, augment=True)
dev_dataset=W2VDataset(dev_protocol, dev_flac_dir, augment=False)
eval_dataset=W2VDataset(eval_protocol, eval_flac_dir, augment=False)

train_loader=DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
dev_loader=DataLoader(dev_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
eval_loader=DataLoader(eval_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

print("Train samples:", len(train_dataset.samples))
print("Bonafide:", sum(1 for _, l in train_dataset.samples if l == 0))
print("Spoof:", sum(1 for _, l in train_dataset.samples if l == 1))
print("Dev samples:", len(dev_dataset.samples))
print("Eval samples:", len(eval_dataset.samples))
print("Sample shape:", train_dataset[0][0].shape)
print("Train loader sampler:", train_loader.sampler)

Train samples: 25380
Bonafide: 2580
Spoof: 22800
Dev samples: 24844
Eval samples: 71237
Sample shape: torch.Size([64000])
Train loader sampler: <torch.utils.data.sampler.RandomSampler object at 0x7a3d66784cb0>


In [12]:
n_bonafide = sum(1 for _, l in train_dataset.samples if l == 0)
n_spoof= sum(1 for _, l in train_dataset.samples if l == 1)

pos_weight=torch.tensor([n_spoof / n_bonafide]).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(pos_weight)

tensor([8.8372], device='cuda:0')


<h2>Training</h2>

In [9]:
model=W2VModel().to(device)
model = model.to(device)
optimizer=optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
epochs=15
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

train_losses, val_losses, val_accuracy, val_precision, val_recall, val_f1, val_eer_list=[], [], [], [], [], [], []
val_roc_auc, val_pr_auc = [], []
best_eer=float("inf")
patience=5
epochs_no_improve=0
best_pr_auc = -1
for epoch in range(epochs):
    model.train()
    running_loss=0
    for wav, labels in train_loader:
        wav=wav.to(device)
        labels=labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs=model(wav)
        loss=criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    avg_train_loss=running_loss / len(train_loader)

    model.eval()
    val_loss=0
    all_labels=[]
    all_scores=[]
    with torch.no_grad():
        for wav, labels in dev_loader:
            wav=wav.to(device)
            labels=labels.to(device).unsqueeze(1)
            outputs=model(wav)
            loss=criterion(outputs, labels)
            val_loss+=loss.item()
            scores=torch.sigmoid(outputs).view(-1)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(labels.view(-1).cpu().numpy())

    avg_val_loss=val_loss/len(dev_loader)
    all_scores=np.array(all_scores)
    all_labels=np.array(all_labels)
    pr_auc = average_precision_score(all_labels, all_scores)
    precision_arr, recall_arr, thresholds = precision_recall_curve(all_labels, all_scores)

    valid = np.where(recall_arr[:-1] >= 0.90)[0]

    if len(valid)>0:
        best_idx=valid[np.argmax(precision_arr[valid])]
        threshold=thresholds[best_idx]
    else:
        threshold=0.5
    
    fpr,tpr,thresholds=roc_curve(all_labels, all_scores)
    fnr=1-tpr
    eer_index=np.nanargmin(np.absolute(fnr - fpr))
    eer=fpr[eer_index]
    eer_threshold=thresholds[eer_index]
    preds=(all_scores > threshold).astype(int)

    accuracy=accuracy_score(all_labels, preds)
    precision= precision_score(all_labels, preds,zero_division=0)
    recall= recall_score(all_labels, preds)
    f1=f1_score(all_labels, preds)
    cm= confusion_matrix(all_labels, preds)
    roc_auc=roc_auc_score(all_labels, all_scores)
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_accuracy.append(accuracy)
    val_precision.append(precision)
    val_recall.append(recall)
    val_f1.append(f1)
    val_eer_list.append(eer)
    val_roc_auc.append(roc_auc)
    val_pr_auc.append(pr_auc)
    
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("Train Loss:", avg_train_loss)
    print("Val Loss:", avg_val_loss)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("EER:", eer)
    print(f"Scores — min:{all_scores.min():.3f} max:{all_scores.max():.3f} "
          f"mean:{all_scores.mean():.3f} std:{all_scores.std():.3f}")
    if eer<best_eer:
        best_eer=eer
    if pr_auc > best_pr_auc:
        best_pr_auc= pr_auc
        epochs_no_improve = 0
        torch.save({
            "epoch":epoch + 1,
            "model_state_dict":model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "accuracy":accuracy,
            "precision":precision,
            "recall":recall,
            "f1":f1,
            "eer":eer,
            "best eer so far":best_eer,
            "eer_threshold":eer_threshold,
            "recall_threshold": threshold,
            "threshold_used":"recall_constrained",
            "pr_auc": pr_auc,
            "roc_auc": roc_auc,
            "train_loss":avg_train_loss,
            "val_loss":avg_val_loss,
            "confusion_matrix":cm.tolist()
        }, "/kaggle/working/audio_w2v.pth")
        print("Best model saved")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve}/{patience} epochs")
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}, best EER: {best_eer:.4f}")
            break
    scheduler.step()
    with open("/kaggle/working/audio_w2v_history.json", "w") as f:
        json.dump({
            "train_loss":train_losses,
            "val_loss":val_losses,
            "val_accuracy":val_accuracy,
            "val_precision":val_precision,
            "val_recall":val_recall,
            "val_f1":val_f1,
            "val_eer":val_eer_list,
            "val_roc_auc": val_roc_auc,
            "val_pr_auc": val_pr_auc
        }, f)
print("Training complete")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/15
Train Loss: 1.9688764957460438
Val Loss: 1.8121627998578496
Accuracy: 0.9083883432619546
Precision: 0.9975149105367793
Recall: 0.9001614639397201
F1: 0.9463410033949453
EER: 0.046703296703296704
Scores — min:0.002 max:0.993 mean:0.767 std:0.314
Best model saved

Epoch 2/15
Train Loss: 0.435048956861922
Val Loss: 0.6890478114485357
Accuracy: 0.9122524553212044
Precision: 0.9995033770361541
Recall: 0.9026731252242555
F1: 0.9486236802413273
EER: 0.016875981161695447
Scores — min:0.007 max:0.998 mean:0.855 std:0.318
Best model saved

Epoch 3/15
Train Loss: 0.1795036760233683
Val Loss: 0.0832957289042521
Accuracy: 0.9178876187409435
Precision: 0.9997532813579394
Recall: 0.9087280229637603
F1: 0.952069921526244
EER: 0.01216640502354788
Scores — min:0.010 max:1.000 mean:0.898 std:0.289
No improvement for 1/5 epochs

Epoch 4/15
Train Loss: 0.09729624985903032
Val Loss: 0.11307479013765023
Accuracy: 0.9188133955884721
Precision: 0.9999013952571119
Recall: 0.9096250448510944
F1: 0.95

KeyboardInterrupt: 

In [13]:
checkpoint = torch.load("/kaggle/input/models/rudranshiverma/w2v-audio-v2/pytorch/default/1/audio_w2v (1).pth", map_location=device, weights_only=False)
model = W2VModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_labels, all_scores = [], []
with torch.no_grad():
    for wav, labels in dev_loader:
        wav = wav.to(device)
        scores = torch.sigmoid(model(wav)).view(-1)
        all_scores.extend(scores.cpu().numpy())
        all_labels.extend(labels.numpy())

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

print(f"{'Thresh':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Missed':>8}")
best_thresh, best_recall = 0.5, 0.0

for t in np.arange(0.05, 0.96, 0.01):
    preds= (all_scores > t).astype(int)
    prec= precision_score(all_labels, preds, zero_division=0)
    rec= recall_score(all_labels, preds, zero_division=0)
    f1= f1_score(all_labels, preds, zero_division=0)
    missed = confusion_matrix(all_labels, preds)[1][0]
    print(f"{t:>8.2f} {prec:>8.4f} {rec:>8.4f} {f1:>8.4f} {missed:>8}")
    if prec >= 0.99 and rec > best_recall:
        best_recall = rec
        best_thresh = t

print(f"\nBest threshold: {best_thresh:.2f}  Recall: {best_recall:.4f}")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Thresh     Prec   Recall       F1   Missed
    0.05   0.9981   0.9985   0.9983       34
    0.06   0.9982   0.9983   0.9983       37
    0.07   0.9983   0.9981   0.9982       43
    0.08   0.9984   0.9978   0.9981       48
    0.09   0.9985   0.9975   0.9980       55
    0.10   0.9985   0.9973   0.9979       61
    0.11   0.9985   0.9970   0.9977       68
    0.12   0.9985   0.9968   0.9976       72
    0.13   0.9986   0.9966   0.9976       76
    0.14   0.9986   0.9965   0.9975       78
    0.15   0.9986   0.9965   0.9975       79
    0.16   0.9986   0.9964   0.9975       80
    0.17   0.9986   0.9962   0.9974       85
    0.18   0.9986   0.9960   0.9973       90
    0.19   0.9986   0.9958   0.9972       93
    0.20   0.9986   0.9957   0.9971       96
    0.21   0.9987   0.9955   0.9971      101
    0.22   0.9986   0.9953   0.9970      104
    0.23   0.9987   0.9953   0.9970      105
    0.24   0.9987   0.9952   0.9969      107
    0.25   0.9987   0.9952   0.9969      107
    0.26  

<h2>Test</h2>

In [14]:
checkpoint = torch.load("/kaggle/input/models/rudranshiverma/w2v-audio-v2/pytorch/default/1/audio_w2v (1).pth", map_location=device, weights_only=False)

model_w2v = W2VModel().to(device)
model_w2v.load_state_dict(checkpoint["model_state_dict"])
model_w2v.eval()

threshold=0.05

print("Checkpoint epoch:", checkpoint["epoch"])
print("Dev EER at save:", checkpoint["eer"])
print("Threshold (recall-based):", threshold)

all_labels = []
all_scores = []

with torch.no_grad():
    for inputs, labels in eval_loader:
        inputs = inputs.to(device)
        outputs = model_w2v(inputs)
        scores = torch.sigmoid(outputs).view(-1)

        all_scores.extend(scores.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

preds = (all_scores > threshold).astype(int)

accuracy = accuracy_score(all_labels, preds)
precision = precision_score(all_labels, preds, zero_division=0)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)
roc_auc = roc_auc_score(all_labels, all_scores)
pr_auc = average_precision_score(all_labels, all_scores)
fpr, tpr, _ = roc_curve(all_labels, all_scores)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

cm = confusion_matrix(all_labels, preds)

print("\nEval Results")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("EER:", eer)
print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print("Threshold used:", threshold)
print("Confusion Matrix:\n", cm)

eval_metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "eer": float(eer),
    "threshold_used": float(threshold),
    "threshold_source": "recall_constrained",
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "confusion_matrix": cm.tolist()
}

with open("/kaggle/working/audio_w2v_eval_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=4)

np.save("/kaggle/working/audio_w2v_fpr.npy", fpr)
np.save("/kaggle/working/audio_w2v_tpr.npy", tpr)

print("Eval metrics saved.")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Checkpoint epoch: 10
Dev EER at save: 0.008634222919937205
Threshold (recall-based): 0.05

Eval Results
Accuracy: 0.9616772182994792
Precision: 0.997850722938648
Recall: 0.9593312670235747
F1: 0.9782119427285352
EER: 0.02583276682528892
ROC-AUC: 0.9962843180591442
PR-AUC: 0.9995135031470108
Threshold used: 0.05
Confusion Matrix:
 [[ 7223   132]
 [ 2598 61284]]
Eval metrics saved.
